# TASK 3A – Fragceylon Dataset Audit and Data Cleaning

**Project:** Retail Product Demand Analysis at Fragceylon  
**Raw dataset:** `Fragceylon_Sales_Dataset.xlsx`

This notebook performs the full dataset audit and cleaning stage before descriptive analysis, statistical inference, or predictive modelling.

The workflow:
1. imports the required Python libraries;
2. loads and inspects the original workbook;
3. checks data types, dates, missing values, duplicates, and invalid values;
4. validates Free Samples, product/category consistency, pricing rules, and Total Value calculations;
5. identifies Quantity outliers without automatically deleting genuine business observations;
6. creates derived date and business variables;
7. separates Regular Orders and Free Samples;
8. exports and verifies the cleaned datasets.

The original raw Excel file is not overwritten.


## 1. Import Required Libraries


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 2. Define the Raw Data File Path



In [2]:
raw_file = Path("../../03_Raw_Data/Fragceylon_Sales_Dataset.xlsx")

print(raw_file.exists())

True


## 3. Inspect the Excel Workbook Structure



In [3]:
excel_file = pd.ExcelFile(raw_file)

print(excel_file.sheet_names)

['Sales Data', 'Summary']


## 4. Load the Sales Data and Confirm Dataset Size




In [4]:
df = pd.read_excel(
    raw_file,
    sheet_name="Sales Data"
)

print(df.shape)

(5481, 11)


In [5]:
print(df.columns.tolist())

df.head()

['Order ID', 'Date', 'Product Name', 'Product Code', 'Channel Type', 'Buyer', 'Line Type', 'Quantity', 'Unit Price (LKR)', 'Total Value (LKR)', 'Export Market']


,Order ID,Date,Product Name,Product Code,Channel Type,Buyer,Line Type,Quantity,Unit Price (LKR),Total Value (LKR),Export Market
0,FC-00001,2024-10-18,Kennedy,003B,Shop,Shop_04,Regular Order,23,450.0,10350.0,Local
1,FC-00002,2024-10-18,Secret Ambrosia,008A,Shop,Shop_02,Regular Order,31,450.0,13950.0,Local
2,FC-00003,2024-10-18,Mesmerose,006B,Shop,Shop_15,Regular Order,35,450.0,15750.0,Local
3,FC-00004,2024-10-20,Mesmerose,006B,Shop,Shop_23,Regular Order,34,450.0,15300.0,Local
4,FC-00005,2024-10-20,Kennedy,003B,Small Dealer,Ruwan Beauty Studio Gampaha,Regular Order,118,348.0,41064.0,Local


## 5. Inspect Data Types and Initial Structure



In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5481 entries, 0 to 5480
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Order ID           5481 non-null   object        
 1   Date               5481 non-null   datetime64[ns]
 2   Product Name       5481 non-null   object        
 3   Product Code       5481 non-null   object        
 4   Channel Type       5481 non-null   object        
 5   Buyer              5481 non-null   object        
 6   Line Type          5481 non-null   object        
 7   Quantity           5481 non-null   int64         
 8   Unit Price (LKR)   5481 non-null   float64       
 9   Total Value (LKR)  5481 non-null   float64       
 10  Export Market      5481 non-null   object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(7)
memory usage: 471.2+ KB


In [7]:
print(df.dtypes)

Order ID                     object
Date                 datetime64[ns]
Product Name                 object
Product Code                 object
Channel Type                 object
Buyer                        object
Line Type                    object
Quantity                      int64
Unit Price (LKR)            float64
Total Value (LKR)           float64
Export Market                object
dtype: object


## 6. Date Validation




In [8]:
date_check = pd.to_datetime(df["Date"], errors="coerce")

invalid_dates = date_check.isna().sum()

print("Invalid dates:", invalid_dates)

Invalid dates: 0


In [9]:
print("Earliest date:", date_check.min())
print("Latest date:", date_check.max())

Earliest date: 2024-10-18 00:00:00
Latest date: 2026-06-30 00:00:00


In [10]:
print("Unique transaction dates:", date_check.nunique())

Unique transaction dates: 474


## 7. Create a Separate Working DataFrame



In [11]:
df_clean = df.copy()

In [12]:
df_clean["Date"] = pd.to_datetime(
    df_clean["Date"],
    errors="coerce"
)

In [13]:
print(df_clean["Date"].dtype)

datetime64[ns]


In [14]:
print("Invalid dates:", df_clean["Date"].isna().sum())
print("Earliest date:", df_clean["Date"].min().date())
print("Latest date:", df_clean["Date"].max().date())
print("Unique transaction dates:", df_clean["Date"].nunique())

Invalid dates: 0
Earliest date: 2024-10-18
Latest date: 2026-06-30
Unique transaction dates: 474


## 8. Missing-Value Analysis


In [15]:
missing_table = pd.DataFrame({
    "Missing Count": df_clean.isna().sum(),
    "Missing Percentage": (df_clean.isna().sum() / len(df_clean) * 100).round(2)
})

missing_table

,Missing Count,Missing Percentage
Order ID,0,0.0
Date,0,0.0
Product Name,0,0.0
Product Code,0,0.0
Channel Type,0,0.0
Buyer,0,0.0
Line Type,0,0.0
Quantity,0,0.0
Unit Price (LKR),0,0.0
Total Value (LKR),0,0.0


In [16]:
print("Total missing values in dataset:", df_clean.isna().sum().sum())

Total missing values in dataset: 0


## 9. Duplicate and Order-ID Audit




In [17]:
exact_duplicates = df_clean.duplicated().sum()

print("Exact duplicate rows:", exact_duplicates)

Exact duplicate rows: 0


In [18]:
duplicate_order_ids = df_clean["Order ID"].duplicated().sum()

print("Duplicate Order IDs:", duplicate_order_ids)
print("Unique Order IDs:", df_clean["Order ID"].nunique())
print("Total rows:", len(df_clean))

Duplicate Order IDs: 0
Unique Order IDs: 5481
Total rows: 5481


In [19]:
print("First Order ID:", df_clean["Order ID"].iloc[0])
print("Last Order ID:", df_clean["Order ID"].iloc[-1])

First Order ID: FC-00001
Last Order ID: FC-05481


In [20]:
invalid_order_id_format = ~df_clean["Order ID"].str.match(r"^FC-\d{5}$")

print("Invalid Order ID formats:", invalid_order_id_format.sum())

Invalid Order ID formats: 0


In [21]:
columns_without_id = [
    col for col in df_clean.columns
    if col != "Order ID"
]

In [22]:
duplicate_like = df_clean[
    df_clean.duplicated(
        subset=columns_without_id,
        keep=False
    )
].sort_values(columns_without_id)

In [23]:
print("Duplicate-looking rows excluding Order ID:", len(duplicate_like))

Duplicate-looking rows excluding Order ID: 604


In [24]:
regular_orders_temp = df_clean[
    df_clean["Line Type"] == "Regular Order"
].copy()

In [25]:
regular_duplicate_like = regular_orders_temp[
    regular_orders_temp.duplicated(
        subset=columns_without_id,
        keep=False
    )
].sort_values(columns_without_id)

print(
    "Duplicate-looking Regular Order rows:",
    len(regular_duplicate_like)
)

Duplicate-looking Regular Order rows: 2


In [26]:
regular_duplicate_like

,Order ID,Date,Product Name,Product Code,Channel Type,Buyer,Line Type,Quantity,Unit Price (LKR),Total Value (LKR),Export Market
985,FC-00986,2025-01-17,Mesmerose,006B,Shop,Shop_21,Regular Order,36,427.5,15390.0,Local
992,FC-00993,2025-01-17,Mesmerose,006B,Shop,Shop_21,Regular Order,36,427.5,15390.0,Local


## 10. Check Zero, Negative, and Invalid Numeric Values



In [27]:
print("Quantity = 0:", (df_clean["Quantity"] == 0).sum())
print("Quantity < 0:", (df_clean["Quantity"] < 0).sum())

print("Unit Price = 0:", (df_clean["Unit Price (LKR)"] == 0).sum())
print("Unit Price < 0:", (df_clean["Unit Price (LKR)"] < 0).sum())

print("Total Value = 0:", (df_clean["Total Value (LKR)"] == 0).sum())
print("Total Value < 0:", (df_clean["Total Value (LKR)"] < 0).sum())

Quantity = 0: 0
Quantity < 0: 0
Unit Price = 0: 0
Unit Price < 0: 0
Total Value = 0: 2365
Total Value < 0: 0


## 11. Validate Regular Orders and Free Samples




In [28]:
print(df_clean["Line Type"].value_counts())

Line Type
Regular Order    3116
Free Sample      2365
Name: count, dtype: int64


In [29]:
zero_value_rows = df_clean[
    df_clean["Total Value (LKR)"] == 0
]

print(zero_value_rows["Line Type"].value_counts())

Line Type
Free Sample    2365
Name: count, dtype: int64


In [30]:
free_samples_temp = df_clean[
    df_clean["Line Type"] == "Free Sample"
]

print(free_samples_temp["Quantity"].value_counts())

Quantity
1    2365
Name: count, dtype: int64


In [31]:
print(
    pd.crosstab(
        df_clean["Channel Type"],
        df_clean["Line Type"]
    )
)

Line Type     Free Sample  Regular Order
Channel Type                            
Main Dealer          2014           2014
Shop                    0            751
Small Dealer          351            351


In [32]:
total_quantity = df_clean["Quantity"].sum()

regular_quantity = df_clean.loc[
    df_clean["Line Type"] == "Regular Order",
    "Quantity"
].sum()

sample_quantity = df_clean.loc[
    df_clean["Line Type"] == "Free Sample",
    "Quantity"
].sum()

print("Total raw quantity:", total_quantity)
print("Regular Order quantity:", regular_quantity)
print("Free Sample quantity:", sample_quantity)
print("Check:", regular_quantity + sample_quantity)

Total raw quantity: 708175
Regular Order quantity: 705810
Free Sample quantity: 2365
Check: 708175


## 12. Product, Channel, Market, and Buyer Consistency




In [33]:
print(df_clean["Product Name"].value_counts())

Product Name
Mesmerose          1692
Kennedy            1505
Dark Matrix        1185
Secret Ambrosia    1099
Name: count, dtype: int64


In [34]:
print("Unique products:", df_clean["Product Name"].nunique())
print(df_clean["Product Name"].unique())

Unique products: 4
['Kennedy' 'Secret Ambrosia' 'Mesmerose' 'Dark Matrix']


In [35]:
print("Unique product codes:", df_clean["Product Code"].nunique())
print(df_clean["Product Code"].value_counts())

Unique product codes: 4
Product Code
006B    1692
003B    1505
009A    1185
008A    1099
Name: count, dtype: int64


In [36]:
product_code_check = (
    df_clean[
        ["Product Name", "Product Code"]
    ]
    .drop_duplicates()
    .sort_values("Product Name")
)

product_code_check

,Product Name,Product Code
7,Dark Matrix,009A
0,Kennedy,003B
2,Mesmerose,006B
1,Secret Ambrosia,008A


In [37]:
print(df_clean["Channel Type"].value_counts())

Channel Type
Main Dealer     4028
Shop             751
Small Dealer     702
Name: count, dtype: int64


In [38]:
print("Unique channels:", df_clean["Channel Type"].nunique())

Unique channels: 3


In [39]:
print(df_clean["Export Market"].value_counts())

Export Market
Local    5111
Japan     370
Name: count, dtype: int64


In [40]:
print("Unique markets:", df_clean["Export Market"].nunique())

Unique markets: 2


In [41]:
print("Unique buyers:", df_clean["Buyer"].nunique())

Unique buyers: 49


In [42]:
sorted(df_clean["Buyer"].unique())

['Asha Saloon Kandy',
 'Diya Spa & Saloon Kurunegala',
 'Dulanji',
 'Kumari Saloon Matara',
 'Mr. Sunil Kumara',
 'Nayana Hair & Beauty Negombo',
 'Oshadi Beauty Point Galle',
 'Prabath Liyanarachchi',
 'Ravin',
 'Ruwan Beauty Studio Gampaha',
 'Sajith',
 'Saloon Shami Maharagama',
 'Sandun Saloon Kalutara',
 'Shop_01',
 'Shop_02',
 'Shop_03',
 'Shop_04',
 'Shop_05',
 'Shop_06',
 'Shop_07',
 'Shop_08',
 'Shop_09',
 'Shop_10',
 'Shop_11',
 'Shop_12',
 'Shop_13',
 'Shop_14',
 'Shop_15',
 'Shop_16',
 'Shop_17',
 'Shop_18',
 'Shop_19',
 'Shop_20',
 'Shop_21',
 'Shop_22',
 'Shop_23',
 'Shop_24',
 'Shop_25',
 'Shop_26',
 'Shop_27',
 'Shop_28',
 'Shop_29',
 'Shop_30',
 'Shop_31',
 'Shop_32',
 'Shop_33',
 'Shop_34',
 'Shop_35',
 'Sriyani Rajapaksha']

In [43]:
text_columns = [
    "Order ID",
    "Product Name",
    "Product Code",
    "Channel Type",
    "Buyer",
    "Line Type",
    "Export Market"
]

for col in text_columns:
    space_issues = (
        df_clean[col].astype(str)
        != df_clean[col].astype(str).str.strip()
    ).sum()

    print(col, ":", space_issues)

Order ID : 0
Product Name : 0
Product Code : 0
Channel Type : 0
Buyer : 0
Line Type : 0
Export Market : 0


In [44]:
for col in [
    "Product Name",
    "Product Code",
    "Channel Type",
    "Line Type",
    "Export Market"
]:
    original_unique = df_clean[col].nunique()

    lowercase_unique = (
        df_clean[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .nunique()
    )

    print(
        col,
        "| Original unique:", original_unique,
        "| After lowercase:", lowercase_unique
    )

Product Name | Original unique: 4 | After lowercase: 4
Product Code | Original unique: 4 | After lowercase: 4
Channel Type | Original unique: 3 | After lowercase: 3
Line Type | Original unique: 2 | After lowercase: 2
Export Market | Original unique: 2 | After lowercase: 2


## 13. Validate the Business Pricing Rules



In [45]:
summary_df = pd.read_excel(
    raw_file,
    sheet_name="Summary",
    header=None
)

summary_df

,0,1,2,3
0,Fragceylon — Sales Summary,NaN,NaN,NaN
1,Period: 2024-10-18 to 2026-06-30,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,By Product,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN
5,Product,Orders (rows),Total Quantity,Total Value (LKR)
6,Mesmerose,1692,216583,75529324.6
7,Kennedy,1505,194586,67786469.5
8,Secret Ambrosia,1099,142371,49571160.3
9,Dark Matrix,1185,154635,53821303.4


In [46]:
print(
    sorted(df_clean["Unit Price (LKR)"].unique())
)

[330.6, 348.0, 361.0, 380.0, 427.5, 450.0]


In [47]:
df_clean["Expected_Unit_Price"] = np.where(
    df_clean["Quantity"] > 100,
    348.00,
    np.where(
        df_clean["Channel Type"].isin(["Shop", "Small Dealer"]),
        450.00,
        np.where(
            df_clean["Channel Type"] == "Main Dealer",
            380.00,
            np.nan
        )
    )
)

In [48]:
january_mask = df_clean["Date"].dt.month == 1

df_clean.loc[
    january_mask,
    "Expected_Unit_Price"
] = (
    df_clean.loc[
        january_mask,
        "Expected_Unit_Price"
    ] * 0.95
)

In [49]:
df_clean["Expected_Unit_Price"] = (
    df_clean["Expected_Unit_Price"].round(2)
)

In [50]:
df_clean["Price_Match"] = np.isclose(
    df_clean["Unit Price (LKR)"],
    df_clean["Expected_Unit_Price"],
    atol=0.01
)

In [51]:
print(df_clean["Price_Match"].value_counts())

Price_Match
True    5481
Name: count, dtype: int64


In [52]:
price_mismatches = df_clean[
    ~df_clean["Price_Match"]
]

print("Pricing-rule mismatches:", len(price_mismatches))

Pricing-rule mismatches: 0


## 14. Validate Total Value and Calculate Commercial Revenue


In [53]:
df_clean["Expected_Total_Value"] = (
    df_clean["Quantity"] * df_clean["Unit Price (LKR)"]
).round(2)

In [54]:
regular_mask = df_clean["Line Type"] == "Regular Order"

In [55]:
df_clean["Total_Value_Match"] = True

df_clean.loc[
    regular_mask,
    "Total_Value_Match"
] = np.isclose(
    df_clean.loc[regular_mask, "Total Value (LKR)"],
    df_clean.loc[regular_mask, "Expected_Total_Value"],
    atol=0.01
)

In [56]:
print(
    df_clean.loc[
        regular_mask,
        "Total_Value_Match"
    ].value_counts()
)

Total_Value_Match
True    3116
Name: count, dtype: int64


In [57]:
regular_value_mismatches = df_clean[
    regular_mask &
    (~df_clean["Total_Value_Match"])
]

print(
    "Regular Order Total Value mismatches:",
    len(regular_value_mismatches)
)

Regular Order Total Value mismatches: 0


In [58]:
free_sample_mask = df_clean["Line Type"] == "Free Sample"

print(
    "Free Samples:",
    free_sample_mask.sum()
)

print(
    "Free Samples with Total Value = 0:",
    (
        df_clean.loc[
            free_sample_mask,
            "Total Value (LKR)"
        ] == 0
    ).sum()
)

Free Samples: 2365
Free Samples with Total Value = 0: 2365


In [59]:
invalid_free_sample_values = df_clean[
    free_sample_mask &
    (df_clean["Total Value (LKR)"] != 0)
]

print(
    "Free Samples with non-zero Total Value:",
    len(invalid_free_sample_values)
)

Free Samples with non-zero Total Value: 0


In [60]:
regular_revenue = df_clean.loc[
    regular_mask,
    "Total Value (LKR)"
].sum()

print(
    "Total Regular Order Revenue:",
    f"LKR {regular_revenue:,.2f}"
)

Total Regular Order Revenue: LKR 246,708,257.80


## 15. Outlier Analysis for Regular Order Quantity




In [61]:
regular_orders = df_clean[
    df_clean["Line Type"] == "Regular Order"
].copy()

In [62]:
print("Regular Orders:", len(regular_orders))

Regular Orders: 3116


In [63]:
quantity_summary = regular_orders["Quantity"].describe()

print(quantity_summary)

count    3116.000000
mean      226.511553
std       183.878177
min        23.000000
25%        55.000000
50%       199.500000
75%       354.000000
max       998.000000
Name: Quantity, dtype: float64


In [64]:
Q1 = regular_orders["Quantity"].quantile(0.25)
Q3 = regular_orders["Quantity"].quantile(0.75)

IQR = Q3 - Q1

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)

Q1: 55.0
Q3: 354.0
IQR: 299.0


In [65]:
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

print("Lower fence:", lower_fence)
print("Upper fence:", upper_fence)

Lower fence: -393.5
Upper fence: 802.5


In [66]:
quantity_outliers = regular_orders[
    (regular_orders["Quantity"] < lower_fence)
    |
    (regular_orders["Quantity"] > upper_fence)
].copy()

In [67]:
print("Number of Quantity outliers:", len(quantity_outliers))

Number of Quantity outliers: 41


In [68]:
print(
    "Smallest outlier quantity:",
    quantity_outliers["Quantity"].min()
)

print(
    "Largest outlier quantity:",
    quantity_outliers["Quantity"].max()
)

Smallest outlier quantity: 810
Largest outlier quantity: 998


In [69]:
quantity_outliers[
    [
        "Order ID",
        "Date",
        "Product Name",
        "Channel Type",
        "Buyer",
        "Quantity",
        "Unit Price (LKR)",
        "Total Value (LKR)",
        "Export Market"
    ]
].head(10)

,Order ID,Date,Product Name,Channel Type,Buyer,Quantity,Unit Price (LKR),Total Value (LKR),Export Market
1575,FC-01576,2025-04-06,Secret Ambrosia,Main Dealer,Prabath Liyanarachchi,910,348.0,316680.0,Japan
1599,FC-01600,2025-04-06,Kennedy,Main Dealer,Prabath Liyanarachchi,810,348.0,281880.0,Japan
1605,FC-01606,2025-04-06,Dark Matrix,Main Dealer,Prabath Liyanarachchi,831,348.0,289188.0,Japan
1655,FC-01656,2025-04-08,Mesmerose,Main Dealer,Prabath Liyanarachchi,927,348.0,322596.0,Japan
1659,FC-01660,2025-04-09,Kennedy,Main Dealer,Prabath Liyanarachchi,981,348.0,341388.0,Japan
1693,FC-01694,2025-04-10,Secret Ambrosia,Main Dealer,Prabath Liyanarachchi,852,348.0,296496.0,Japan
1751,FC-01752,2025-04-13,Kennedy,Main Dealer,Prabath Liyanarachchi,896,348.0,311808.0,Japan
1821,FC-01822,2025-04-17,Mesmerose,Main Dealer,Prabath Liyanarachchi,946,348.0,329208.0,Japan
1873,FC-01874,2025-04-19,Secret Ambrosia,Main Dealer,Prabath Liyanarachchi,989,348.0,344172.0,Japan
1923,FC-01924,2025-04-21,Mesmerose,Main Dealer,Prabath Liyanarachchi,893,348.0,310764.0,Japan


In [70]:
print(quantity_outliers["Channel Type"].value_counts())

Channel Type
Main Dealer    41
Name: count, dtype: int64


In [71]:
print(quantity_outliers["Export Market"].value_counts())

Export Market
Japan    41
Name: count, dtype: int64


In [72]:
print(quantity_outliers["Buyer"].value_counts())

Buyer
Prabath Liyanarachchi    41
Name: count, dtype: int64


In [73]:
print(quantity_outliers["Product Name"].value_counts())

Product Name
Kennedy            12
Mesmerose          12
Secret Ambrosia    10
Dark Matrix         7
Name: count, dtype: int64


## 16. Create an Outlier Flag



In [74]:
df_clean["Outlier_IQR_Flag"] = "No"

In [75]:
df_clean.loc[
    (df_clean["Line Type"] == "Regular Order")
    &
    (
        (df_clean["Quantity"] < lower_fence)
        |
        (df_clean["Quantity"] > upper_fence)
    ),
    "Outlier_IQR_Flag"
] = "Yes"

In [76]:
print(df_clean["Outlier_IQR_Flag"].value_counts())

Outlier_IQR_Flag
No     5440
Yes      41
Name: count, dtype: int64


## 17. Create Derived Date Variables




In [77]:
df_clean["Year"] = df_clean["Date"].dt.year

In [78]:
print(df_clean["Year"].value_counts().sort_index())

Year
2024     730
2025    2829
2026    1922
Name: count, dtype: int64


In [79]:
df_clean["Month"] = df_clean["Date"].dt.month_name()

In [80]:
print(df_clean["Month"].value_counts())

Month
April        1334
December     1290
January      1158
November      495
March         481
February      176
May           156
June          128
October       123
September      76
August         46
July           18
Name: count, dtype: int64


In [81]:
df_clean["Month_Number"] = df_clean["Date"].dt.month

In [82]:
df_clean["Quarter"] = "Q" + df_clean["Date"].dt.quarter.astype(str)

In [83]:
print(df_clean["Quarter"].value_counts())

Quarter
Q4    1908
Q1    1815
Q2    1618
Q3     140
Name: count, dtype: int64


In [84]:
df_clean["Day_of_Week"] = df_clean["Date"].dt.day_name()

In [85]:
print(df_clean["Day_of_Week"].value_counts())

Day_of_Week
Sunday       879
Wednesday    824
Saturday     791
Monday       769
Thursday     768
Friday       728
Tuesday      722
Name: count, dtype: int64


In [86]:
df_clean["Year_Month"] = (
    df_clean["Date"]
    .dt.to_period("M")
    .astype(str)
)

In [87]:
print(df_clean["Year_Month"].unique()[:10])

['2024-10' '2024-11' '2024-12' '2025-01' '2025-02' '2025-03' '2025-04'
 '2025-05' '2025-06' '2025-07']


In [88]:
start_date = df_clean["Date"].min()

df_clean["Time_Trend"] = (
    df_clean["Date"] - start_date
).dt.days

In [89]:
print("Minimum Time Trend:", df_clean["Time_Trend"].min())
print("Maximum Time Trend:", df_clean["Time_Trend"].max())

Minimum Time Trend: 0
Maximum Time Trend: 620


## 18. Create the Sales Regime Variable



In [90]:
regime_change_date = pd.Timestamp("2025-04-01")

df_clean["Sales_Regime"] = np.where(
    df_clean["Date"] < regime_change_date,
    "Pre-April-2025",
    "Post-April-2025"
)

In [91]:
print(df_clean["Sales_Regime"].value_counts())

Sales_Regime
Post-April-2025    4028
Pre-April-2025     1453
Name: count, dtype: int64


In [92]:
regime_channel_table = pd.crosstab(
    df_clean["Sales_Regime"],
    df_clean["Channel Type"]
)

regime_channel_table

Channel Type,Main Dealer,Shop,Small Dealer
Sales_Regime,,,
Post-April-2025,4028,0,0
Pre-April-2025,0,751,702


## 19. Create Market and Order-Type Indicators



In [93]:
df_clean["Market_Group"] = np.where(
    df_clean["Export Market"] == "Local",
    "Local",
    "Export"
)

In [94]:
print(df_clean["Market_Group"].value_counts())

Market_Group
Local     5111
Export     370
Name: count, dtype: int64


In [95]:
df_clean["Is_Regular_Order"] = np.where(
    df_clean["Line Type"] == "Regular Order",
    1,
    0
)

In [96]:
print(df_clean["Is_Regular_Order"].value_counts())

Is_Regular_Order
1    3116
0    2365
Name: count, dtype: int64


## 20. Review the Derived Variables


In [97]:
print(df_clean.columns.tolist())

['Order ID', 'Date', 'Product Name', 'Product Code', 'Channel Type', 'Buyer', 'Line Type', 'Quantity', 'Unit Price (LKR)', 'Total Value (LKR)', 'Export Market', 'Expected_Unit_Price', 'Price_Match', 'Expected_Total_Value', 'Total_Value_Match', 'Outlier_IQR_Flag', 'Year', 'Month', 'Month_Number', 'Quarter', 'Day_of_Week', 'Year_Month', 'Time_Trend', 'Sales_Regime', 'Market_Group', 'Is_Regular_Order']


In [98]:
df_clean[
    [
        "Date",
        "Year",
        "Month",
        "Month_Number",
        "Quarter",
        "Day_of_Week",
        "Year_Month",
        "Time_Trend",
        "Sales_Regime",
        "Market_Group",
        "Line Type",
        "Is_Regular_Order"
    ]
].head(10)

,Date,Year,Month,Month_Number,Quarter,Day_of_Week,Year_Month,Time_Trend,Sales_Regime,Market_Group,Line Type,Is_Regular_Order
0,2024-10-18,2024,October,10,Q4,Friday,2024-10,0,Pre-April-2025,Local,Regular Order,1
1,2024-10-18,2024,October,10,Q4,Friday,2024-10,0,Pre-April-2025,Local,Regular Order,1
2,2024-10-18,2024,October,10,Q4,Friday,2024-10,0,Pre-April-2025,Local,Regular Order,1
3,2024-10-20,2024,October,10,Q4,Sunday,2024-10,2,Pre-April-2025,Local,Regular Order,1
4,2024-10-20,2024,October,10,Q4,Sunday,2024-10,2,Pre-April-2025,Local,Regular Order,1
5,2024-10-20,2024,October,10,Q4,Sunday,2024-10,2,Pre-April-2025,Local,Free Sample,0
6,2024-10-20,2024,October,10,Q4,Sunday,2024-10,2,Pre-April-2025,Local,Regular Order,1
7,2024-10-21,2024,October,10,Q4,Monday,2024-10,3,Pre-April-2025,Local,Regular Order,1
8,2024-10-21,2024,October,10,Q4,Monday,2024-10,3,Pre-April-2025,Local,Free Sample,0
9,2024-10-21,2024,October,10,Q4,Monday,2024-10,3,Pre-April-2025,Local,Regular Order,1


## 21. Build the Final Cleaned Master Dataset




In [99]:
audit_columns = [
    "Expected_Unit_Price",
    "Price_Match",
    "Expected_Total_Value",
    "Total_Value_Match"
]

cleaned_master = df_clean.drop(
    columns=audit_columns
).copy()

In [100]:
print("Cleaned Master shape:", cleaned_master.shape)

Cleaned Master shape: (5481, 22)


In [101]:
print(cleaned_master.columns.tolist())

['Order ID', 'Date', 'Product Name', 'Product Code', 'Channel Type', 'Buyer', 'Line Type', 'Quantity', 'Unit Price (LKR)', 'Total Value (LKR)', 'Export Market', 'Outlier_IQR_Flag', 'Year', 'Month', 'Month_Number', 'Quarter', 'Day_of_Week', 'Year_Month', 'Time_Trend', 'Sales_Regime', 'Market_Group', 'Is_Regular_Order']


## 22. Split the Cleaned Dataset




In [102]:
regular_orders_clean = cleaned_master[
    cleaned_master["Line Type"] == "Regular Order"
].copy()

In [103]:
print("Regular Orders rows:", len(regular_orders_clean))

Regular Orders rows: 3116


In [104]:
free_samples_clean = cleaned_master[
    cleaned_master["Line Type"] == "Free Sample"
].copy()

In [105]:
print("Free Sample rows:", len(free_samples_clean))

Free Sample rows: 2365


## 23. Post-Cleaning Verification



In [106]:
print("Cleaned Master:", len(cleaned_master))
print("Regular Orders:", len(regular_orders_clean))
print("Free Samples:", len(free_samples_clean))

print(
    "Regular + Free Samples:",
    len(regular_orders_clean) + len(free_samples_clean)
)

Cleaned Master: 5481
Regular Orders: 3116
Free Samples: 2365
Regular + Free Samples: 5481


In [107]:
print(
    "Master unique Order IDs:",
    cleaned_master["Order ID"].nunique()
)

print(
    "Regular unique Order IDs:",
    regular_orders_clean["Order ID"].nunique()
)

print(
    "Free Sample unique Order IDs:",
    free_samples_clean["Order ID"].nunique()
)

Master unique Order IDs: 5481
Regular unique Order IDs: 3116
Free Sample unique Order IDs: 2365


In [108]:
print(
    regular_orders_clean["Outlier_IQR_Flag"].value_counts()
)

Outlier_IQR_Flag
No     3075
Yes      41
Name: count, dtype: int64


In [109]:
print(
    free_samples_clean["Quantity"].value_counts()
)

print(
    free_samples_clean["Total Value (LKR)"].value_counts()
)

Quantity
1    2365
Name: count, dtype: int64
Total Value (LKR)
0.0    2365
Name: count, dtype: int64


In [110]:
print(
    "Missing values in Cleaned Master:",
    cleaned_master.isna().sum().sum()
)

print(
    "Missing values in Regular Orders:",
    regular_orders_clean.isna().sum().sum()
)

print(
    "Missing values in Free Samples:",
    free_samples_clean.isna().sum().sum()
)

Missing values in Cleaned Master: 0
Missing values in Regular Orders: 0
Missing values in Free Samples: 0


## 24. Export the Cleaned Datasets




In [111]:
cleaned_folder = Path("../../04_Cleaned_Data")

cleaned_folder.mkdir(
    parents=True,
    exist_ok=True
)

In [112]:
cleaned_master.to_excel(
    cleaned_folder / "Fragceylon_Cleaned_Master.xlsx",
    index=False
)

In [113]:
regular_orders_clean.to_excel(
    cleaned_folder / "Fragceylon_Regular_Orders.xlsx",
    index=False
)

In [118]:
free_samples_clean.to_excel(
    cleaned_folder / "Fragceylon_Free_Samples.xlsx",
    index=False
)

In [119]:
for file in cleaned_folder.glob("Fragceylon_*.xlsx"):
    print(file.name)

Fragceylon_Cleaned_Master.xlsx
Fragceylon_Free_Samples.xlsx
Fragceylon_Regular_Orders.xlsx
Fragceylon_Sales_Dataset_Working.xlsx


In [120]:
print(
    (cleaned_folder / "Fragceylon_Cleaned_Master.xlsx").exists()
)

print(
    (cleaned_folder / "Fragceylon_Regular_Orders.xlsx").exists()
)

print(
    (cleaned_folder / "Fragceylon_Free_Samples.xlsx").exists()
)

True
True
True


## 25. Re-Import and Verify the Saved Regular Order Dataset


In [121]:
verification_df = pd.read_excel(
    cleaned_folder / "Fragceylon_Regular_Orders.xlsx"
)

print("Saved Regular Orders rows:", len(verification_df))
print("Saved Regular Orders columns:", verification_df.shape[1])
print("Saved missing values:", verification_df.isna().sum().sum())

Saved Regular Orders rows: 3116
Saved Regular Orders columns: 22
Saved missing values: 0


In [122]:
print(
    "Saved total demand:",
    verification_df["Quantity"].sum()
)

Saved total demand: 705810


## 26. Task 3A Cleaning Conclusion

The full Fragceylon dataset audit and cleaning workflow has been completed successfully.

### Final verified findings

- Raw observations: **5,481**
- Original variables: **11**
- Date range: **18 October 2024 to 30 June 2026**
- Missing values: **0**
- Exact duplicate rows: **0**
- Duplicate Order IDs: **0**
- Pricing-rule mismatches: **0**
- Regular Order Total Value mismatches: **0**
- Regular Orders: **3,116**
- Free Samples: **2,365**
- Commercial demand represented by Regular Order Quantity: **705,810 units**
- Regular Order revenue: **LKR 246,708,257.80**
- High-Quantity IQR outliers retained and flagged: **41**
- Cleaned Master rows: **5,481**
- Saved Regular Order rows: **3,116**
- Saved Free Sample rows: **2,365**

### Final cleaning decisions

- The original raw workbook was preserved.
- No rows were removed for missing values or exact duplication.
- The duplicate-looking Regular Order pair was retained because the Order IDs are unique.
- Free Samples were preserved but separated from commercial demand.
- High-volume Japan orders were retained and flagged rather than automatically removed.
- `Total Value (LKR)` will not be used as a predictor of Quantity because it creates data leakage.
- `Unit Price (LKR)` will be treated cautiously because the pricing rules depend partly on Quantity and Channel.
- `Sales_Regime` was created to represent the structural business/channel change from 1 April 2025.

**Task 3A is complete. The next stage is Task 3 descriptive analysis using the cleaned Regular Orders dataset.**
